<a href="https://colab.research.google.com/github/Lucas-Maingi/seer/blob/main/notebooks/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Seer — cloud training run

Trains all three Seer models on a free GPU (Colab T4 / Kaggle) and hands back
a small zip of deployable ONNX weights for the CPU serving machine.

**Runtime → Change runtime type → T4 GPU** before running.

Everything trained here is synthetic specimen data generated in this session —
nothing sensitive is uploaded or downloaded.

In [ ]:
!nvidia-smi -L

In [ ]:
!git clone https://github.com/Lucas-Maingi/seer.git
%cd seer
!pip install -q -e .

## Generate the synthetic dataset

~8k samples is a good balance for models this small (raise `COUNT` if you have
session time to spare — generation is CPU-bound, ~1.4 s/sample).

Optional realism upgrade for the portraits: upload a folder of synthetic faces
(e.g. a slice of the [SFHQ dataset](https://github.com/SelfishGene/SFHQ-dataset))
and pass `FACES=/content/faces`.

In [ ]:
import os
os.environ["COUNT"] = "8000"
# os.environ["FACES"] = "/content/faces"   # optional synthetic-face dir
!bash scripts/train_all.sh

== 1/6 dataset (8000 samples -> data/synth) ==
generating:   4% 352/8000 [04:35<1:39:49,  1.28it/s]

## Package the deployable weights

Only the ONNX files (and training checkpoints if you want to resume later)
need to leave this machine — a few tens of MB.

In [ ]:
!zip -j seer_weights.zip weights/*.onnx weights/*.pt runs/latency.md
from google.colab import files
files.download("seer_weights.zip")

## Back on the laptop

```bash
unzip seer_weights.zip -d weights/
python scripts/fetch_models.py          # YuNet + ArcFace for the face stage
python -m seer.runtime.bench            # re-bench on the real serving CPU
uvicorn seer.api.main:app --port 8000   # and in web/: npm run dev
```

The latency report that matters is the one measured on the serving hardware,
not this VM — rerun `seer.runtime.bench` locally and commit that.